In [108]:
import numpy as np
from dlfs.helpers import im2col_strided, im2col, col2im_strided
from dlfs.base import Layer

"""X = np.arange(25).reshape(5, 5)
X = np.array([[X]])"""

X = np.random.randn(30, 5, 28, 28)

In [109]:
def maxpool2d(X, pool_shape, stride, padding):
    B, C = X.shape[:2]
    kH, kW = pool_shape
    cols, out_H, out_W = im2col_strided(X, kernel_shape=pool_shape, stride=stride, padding=padding)
    
    # separate channels and flatten window
    cols_reshaped = cols.reshape(B, C, kH*kW, out_H*out_W)
    
    # max over pooling window
    pooled = np.max(cols_reshaped, axis=2)
    
    # reshape to 4D output
    return pooled.reshape(B, C, out_H, out_W)

In [110]:
stride = 1
padding = 0
pool_size = 2

maxed = maxpool2d(X, pool_shape=(pool_size, pool_size), stride=stride, padding=padding)

print(maxed.shape)

(30, 5, 27, 27)


In [111]:
import torch.nn as nn
import torch

X = torch.Tensor(X)

pool = nn.MaxPool2d(kernel_size=pool_size, stride=stride, padding=padding)
y = pool(X)  # no padding

print(y.shape)

print(np.allclose(y.detach().numpy(), maxed))

torch.Size([30, 5, 27, 27])
True


In [112]:
X = np.arange(25).reshape(5, 5)
X = np.array([[X]])

print(X)

stride = 1
padding = 0
pool_size = 2
pool_shape = (pool_size, pool_size)

Y = maxpool2d(X, pool_shape=pool_shape, stride=stride, padding=padding)

print(Y)

[[[[ 0  1  2  3  4]
   [ 5  6  7  8  9]
   [10 11 12 13 14]
   [15 16 17 18 19]
   [20 21 22 23 24]]]]
[[[[ 6  7  8  9]
   [11 12 13 14]
   [16 17 18 19]
   [21 22 23 24]]]]


In [113]:
delta = np.random.randn(*Y.shape)

print(delta)

[[[[-0.14190155 -0.08821515 -0.35165844 -0.85558084]
   [-0.22584436  1.27967852 -0.2666524  -0.01611882]
   [ 1.17636044 -0.29324736  1.1687726   1.0442922 ]
   [ 1.85909962 -0.16461108 -0.29163122  0.20236802]]]]


In [114]:
def maxpool2d_new(X, pool_shape, stride, padding):
    B, C = X.shape[:2]
    kH, kW = pool_shape

    cols, out_H, out_W = im2col_strided(X, kernel_shape=pool_shape,
                                        stride=stride, padding=padding)

    cols_reshaped = cols.reshape(B, C, kH*kW, out_H*out_W)

    # max and argmax along pooling window
    max_vals = np.max(cols_reshaped, axis=2)
    max_idx = np.argmax(cols_reshaped, axis=2)   # <--- NEEDED FOR BACKWARD

    pooled = max_vals.reshape(B, C, out_H, out_W)

    return pooled, (cols_reshaped, max_idx, X.shape, pool_shape, stride, padding)


def maxpool2d_backward(dout, cache):
    cols_reshaped, max_idx, X_shape, pool_shape, stride, padding = cache
    B, C, H, W = X_shape
    kH, kW = pool_shape

    B, C, _, out_HW = cols_reshaped.shape
    out_H = int((out_HW)**0.5)   # or store this in cache explicitly
    out_W = out_H

    # dout flattened to match last dimension (out_H*out_W)
    dout_flat = dout.reshape(B, C, out_HW)

    # Create zero gradient for all col values
    dcols = np.zeros_like(cols_reshaped)

    # Scatter dout to positions of max values
    # max_idx has shape (B,C,out_HW); use advanced indexing
    b_idx = np.arange(B)[:, None, None]
    c_idx = np.arange(C)[None, :, None]
    hw_idx = np.arange(out_HW)[None, None, :]

    dcols[b_idx, c_idx, max_idx, hw_idx] = dout_flat

    # reshape back to original im2col shape
    dcols = dcols.reshape(B, C*kH*kW, out_HW)

    # convert col-gradients back to image
    dX = col2im_strided(dcols, X_shape[-2:], kernel_shape=pool_shape,
                        stride=stride, padding=padding)

    return dX


In [115]:
#X = np.arange(49).reshape(7, 7)
#X = np.array([[X]])
X = np.round(np.random.randn(5, 3, 7, 7), 3)

print(X[0, 0])

stride = 2
padding = 0
pool_size = 3
pool_shape = (2, 2)

Y, cache = maxpool2d_new(X, pool_shape=pool_shape, stride=stride, padding=padding)

print(Y[0, 0])

[[-0.247  0.882  0.027  0.985  1.64  -2.274  0.073]
 [ 0.66  -0.809 -0.77   0.04  -1.024 -0.121 -1.261]
 [-1.929 -1.823  0.153  0.087 -0.852  0.874  0.813]
 [-1.353  0.153 -0.236  0.309 -0.62   0.082 -0.73 ]
 [-0.655 -0.264 -0.681 -2.275 -1.093  0.853  0.328]
 [ 1.009  0.184 -1.102  1.469  1.732 -0.357 -0.73 ]
 [-1.294  0.251  0.604 -0.583 -1.898 -1.352  0.638]]
[[0.882 0.985 1.64 ]
 [0.153 0.309 0.874]
 [1.009 1.469 1.732]]


In [116]:
delta = np.ones_like(Y)

grad = maxpool2d_backward(delta, cache)

print(grad[0, 0])

[[0. 1. 0. 1. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0.]
 [0. 1. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 1. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0.]]


In [117]:
class MaxPoolLayer(Layer):

    def __init__(self, pool_size: int | tuple, stride: int, padding: int):
        self.stride = stride
        self.padding = padding

        if isinstance(pool_size, tuple):
            self.pool_size = pool_size
        else:
            self.pool_size = (pool_size, pool_size)

    def forward(self, inputs):
        self.H_in, self.W_in = inputs.shape[-2:]
        B, C = inputs.shape[:2]
        kH, kW = self.pool_size

        cols, out_H, out_W = im2col_strided(inputs, kernel_shape=self.pool_size,
                                            stride=self.stride, padding=self.padding)

        self.cols_reshaped = cols.reshape(B, C, kH*kW, out_H*out_W)

        max_vals = np.max(self.cols_reshaped, axis=2)
        self.max_idx = np.argmax(self.cols_reshaped, axis=2)   # <--- NEEDED FOR BACKWARD

        pooled = max_vals.reshape(B, C, out_H, out_W)

        self.output = pooled

    def backward(self, delta):
        B, C, _, out_HW = self.cols_reshaped.shape
        kH, kW = self.pool_size

        delta_flat = delta.reshape(B, C, out_HW)
        
        dcols = np.zeros_like(self.cols_reshaped)

        # Scatter dout to positions of max values
        # max_idx has shape (B,C,out_HW); use advanced indexing
        b_idx = np.arange(B)[:, None, None]
        c_idx = np.arange(C)[None, :, None]
        hw_idx = np.arange(out_HW)[None, None, :]

        dcols[b_idx, c_idx, self.max_idx, hw_idx] = delta_flat

        # reshape back to original im2col shape
        dcols = dcols.reshape(B, C*kH*kW, out_HW)

        # convert col-gradients back to image
        dX = col2im_strided(dcols, output_shape=(self.H_in, self.W_in), kernel_shape=self.pool_size,
                            stride=self.stride, padding=self.padding)

        self.dinputs = dX

In [118]:
maxp = MaxPoolLayer(pool_size=2, stride=2, padding=0)

maxp.forward(X)

maxp.backward(delta)

print(np.allclose(maxp.output, Y))
print(np.allclose(maxp.dinputs, grad))

True
True


In [120]:
import torch
import torch.nn as nn

X = torch.tensor(X)

pool = nn.MaxPool2d(kernel_size=2, stride=2, return_indices=True)
y, indices = pool(X)

# suppose we have some custom gradient from next layer
grad_out = torch.tensor(delta)

grad_input = torch.nn.functional.max_unpool2d(
    grad_out, indices, kernel_size=2, stride=2, output_size=X.shape
)

print(np.allclose(grad_input.detach().numpy(), maxp.dinputs))
print(np.allclose(y.detach().numpy(), maxp.output))

True
True


C:\Users\glovric\AppData\Local\Temp\ipykernel_9196\3393903685.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X)
